# Natural Language SQL Insight Agent

Ask a question in plain English. The agent:

1. **Matches** the question to the most relevant pre-written SQL query in the query repository (the LLM picks it from each query's description and example questions).
2. **Executes** that query against a SQLite database, in read-only mode.
3. **Answers** by having the LLM turn the raw rows into a plain-English reply.

This notebook is self-contained. It holds everything that was in the `sql_agent` folder: the sample database, the query repository, the agent, a chat interface, and the tests.

**Before you start:** you need an OpenAI API key. Put it in a `.env` file next to this notebook (copy `.env.example` to `.env` and fill in the value). If no key is found, the setup section asks for it.

## 1. Setup

The `openai` package is already installed in the project's `.venv`, so no install step is needed here.

In [1]:
import getpass
import json
import os
import random
import re
import sqlite3
import tempfile
import unittest
from dataclasses import dataclass
from datetime import date, timedelta
from pathlib import Path
from types import SimpleNamespace

import openai

# A small, inexpensive OpenAI model. Change SQL_AGENT_MODEL to use another model you have access to.
MODEL = os.environ.get("SQL_AGENT_MODEL", "gpt-4o-mini")
DB_PATH = Path.cwd() / "sample.db"
ENV_FILE = Path.cwd() / ".env"
MAX_ROWS_FOR_ANSWER = 50

# Load KEY=value pairs from .env (if present) without overwriting variables already set.
if ENV_FILE.exists():
    for line in ENV_FILE.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            name, value = line.split("=", 1)
            os.environ.setdefault(name.strip(), value.strip().strip('"').strip("'"))

if not os.environ.get("OPENAI_API_KEY"):
    try:
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key: ")
    except Exception:
        # Some frontends (e.g. non-interactive runs) cannot prompt for input.
        pass

print("Model:", MODEL)
print("Database:", DB_PATH)
print("API key:", "set" if os.environ.get("OPENAI_API_KEY") else "NOT set (tests still run; live questions will fail)")

Model: gpt-4o-mini
Database: c:\Users\RAGHAV KRISHNA\Desktop\AI-Agents\sql_agent\sample.db
API key: set


## 2. Sample database

A small retail store with customers, products, orders and order items. The data is generated with a fixed random seed, so every run produces the same database.

In [2]:
SCHEMA = """
CREATE TABLE customers (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    country TEXT NOT NULL,
    signup_date TEXT NOT NULL
);
CREATE TABLE products (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    category TEXT NOT NULL,
    price REAL NOT NULL
);
CREATE TABLE orders (
    id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL REFERENCES customers(id),
    order_date TEXT NOT NULL,
    status TEXT NOT NULL
);
CREATE TABLE order_items (
    id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL REFERENCES orders(id),
    product_id INTEGER NOT NULL REFERENCES products(id),
    quantity INTEGER NOT NULL,
    unit_price REAL NOT NULL
);
"""

COUNTRIES = ["USA", "UK", "India", "Germany", "Canada", "Australia"]
PRODUCTS = [
    ("Wireless Mouse", "Electronics", 25.0),
    ("Mechanical Keyboard", "Electronics", 89.0),
    ("27in Monitor", "Electronics", 240.0),
    ("USB-C Hub", "Electronics", 45.0),
    ("Standing Desk", "Furniture", 420.0),
    ("Ergonomic Chair", "Furniture", 310.0),
    ("Desk Lamp", "Furniture", 38.0),
    ("Notebook Pack", "Stationery", 12.0),
    ("Gel Pens (10)", "Stationery", 9.0),
    ("Whiteboard Markers", "Stationery", 15.0),
]


def build_sample_db(path=DB_PATH, seed=42):
    """(Re)create the sample store database with deterministic random data."""
    rng = random.Random(seed)
    if path.exists():
        path.unlink()

    conn = sqlite3.connect(path)
    conn.executescript(SCHEMA)

    start = date(2025, 1, 1)
    customers = []
    for i in range(1, 121):
        signup = start + timedelta(days=rng.randint(0, 500))
        customers.append((i, f"Customer {i}", rng.choice(COUNTRIES), signup.isoformat()))
    conn.executemany("INSERT INTO customers VALUES (?, ?, ?, ?)", customers)

    products = [(i, name, cat, price) for i, (name, cat, price) in enumerate(PRODUCTS, 1)]
    conn.executemany("INSERT INTO products VALUES (?, ?, ?, ?)", products)

    item_id = 1
    for order_id in range(1, 801):
        order_date = start + timedelta(days=rng.randint(0, 640))
        status = rng.choices(["delivered", "shipped", "cancelled"], weights=[85, 10, 5])[0]
        customer_id = rng.randint(1, 120)
        conn.execute(
            "INSERT INTO orders VALUES (?, ?, ?, ?)",
            (order_id, customer_id, order_date.isoformat(), status),
        )
        for _ in range(rng.randint(1, 4)):
            product_id, _, _, price = rng.choice(products)
            qty = rng.randint(1, 3)
            conn.execute(
                "INSERT INTO order_items VALUES (?, ?, ?, ?, ?)",
                (item_id, order_id, product_id, qty, price),
            )
            item_id += 1

    conn.commit()
    conn.close()
    return path


def run_query(sql, db_path=DB_PATH, max_rows=200):
    """Run a SELECT/WITH query in read-only mode. Returns (column_names, rows)."""
    if not sql.lstrip().lower().startswith(("select", "with")):
        raise ValueError("Only SELECT queries are allowed.")

    # mode=ro makes SQLite refuse any write, even if the SQL were malicious.
    conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    try:
        cursor = conn.execute(sql)
        columns = [desc[0] for desc in cursor.description or []]
        rows = cursor.fetchmany(max_rows)
        return columns, rows
    finally:
        conn.close()


if not DB_PATH.exists():
    build_sample_db()
    print("Created sample database at", DB_PATH)

columns, rows = run_query("SELECT COUNT(*) AS orders FROM orders")
print("Orders in database:", rows[0][0])

Orders in database: 800


## 3. Query repository

Every query the agent is allowed to run is listed here. The agent never writes SQL itself. The LLM only chooses one of these IDs, and the matching SQL is then executed.

Each entry has:
- `id`: a unique name
- `description`: what the query answers
- `example_questions`: the phrasings users might use (these help the matching)
- `sql`: the query

In [3]:
REPOSITORY_JSON = """
{
  "queries": [
    {
      "id": "monthly_revenue",
      "description": "Total revenue per month from delivered orders.",
      "example_questions": ["What was our monthly revenue?", "Show sales by month", "How much did we make each month?"],
      "sql": "SELECT strftime('%Y-%m', o.order_date) AS month, ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue FROM orders o JOIN order_items oi ON oi.order_id = o.id WHERE o.status = 'delivered' GROUP BY month ORDER BY month;"
    },
    {
      "id": "top_products_by_revenue",
      "description": "Top 5 products ranked by revenue from delivered orders.",
      "example_questions": ["What are our best selling products?", "Which products earn the most?", "Top 5 products by revenue"],
      "sql": "SELECT p.name AS product, ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue FROM order_items oi JOIN products p ON p.id = oi.product_id JOIN orders o ON o.id = oi.order_id WHERE o.status = 'delivered' GROUP BY p.id, p.name ORDER BY revenue DESC LIMIT 5;"
    },
    {
      "id": "revenue_by_country",
      "description": "Revenue from delivered orders grouped by customer country.",
      "example_questions": ["Which country buys the most?", "Revenue by country", "Where are our customers located and how much do they spend?"],
      "sql": "SELECT c.country, ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue FROM orders o JOIN customers c ON c.id = o.customer_id JOIN order_items oi ON oi.order_id = o.id WHERE o.status = 'delivered' GROUP BY c.country ORDER BY revenue DESC;"
    },
    {
      "id": "top_customers_by_spend",
      "description": "Top 10 customers by total spend on delivered orders.",
      "example_questions": ["Who are our best customers?", "Who spends the most?", "Top 10 customers by spending"],
      "sql": "SELECT c.name AS customer, c.country, ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_spend FROM customers c JOIN orders o ON o.customer_id = c.id JOIN order_items oi ON oi.order_id = o.id WHERE o.status = 'delivered' GROUP BY c.id, c.name, c.country ORDER BY total_spend DESC LIMIT 10;"
    },
    {
      "id": "average_order_value",
      "description": "Average value of a delivered order.",
      "example_questions": ["What is the average order value?", "How much does a typical order cost?", "Average basket size in money"],
      "sql": "SELECT ROUND(SUM(oi.quantity * oi.unit_price) / COUNT(DISTINCT o.id), 2) AS average_order_value FROM orders o JOIN order_items oi ON oi.order_id = o.id WHERE o.status = 'delivered';"
    },
    {
      "id": "orders_by_status",
      "description": "Number of orders in each status (delivered, shipped, cancelled).",
      "example_questions": ["How many orders are delivered?", "How many orders were cancelled?", "Order status breakdown"],
      "sql": "SELECT status, COUNT(*) AS order_count FROM orders GROUP BY status ORDER BY order_count DESC;"
    },
    {
      "id": "category_sales",
      "description": "Revenue and units sold per product category from delivered orders.",
      "example_questions": ["Which category sells best?", "Sales by product category", "How many units did each category sell?"],
      "sql": "SELECT p.category, SUM(oi.quantity) AS units_sold, ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue FROM order_items oi JOIN products p ON p.id = oi.product_id JOIN orders o ON o.id = oi.order_id WHERE o.status = 'delivered' GROUP BY p.category ORDER BY revenue DESC;"
    },
    {
      "id": "new_customers_by_year",
      "description": "Number of customers who signed up in each year.",
      "example_questions": ["How many new customers did we get each year?", "Customer signups by year", "Customer growth over time"],
      "sql": "SELECT strftime('%Y', signup_date) AS signup_year, COUNT(*) AS new_customers FROM customers GROUP BY signup_year ORDER BY signup_year;"
    },
    {
      "id": "product_catalog",
      "description": "All products with their category and price, cheapest first within each category.",
      "example_questions": ["What products do we sell?", "Show the product catalog", "What is the price of each item?"],
      "sql": "SELECT category, name AS product, price FROM products ORDER BY category, price;"
    }
  ]
}
"""


def load_repository():
    return json.loads(REPOSITORY_JSON)["queries"]


REPOSITORY = load_repository()

# Optional: you can also add queries here at runtime, e.g.
# REPOSITORY.append({"id": "...", "description": "...", "example_questions": [...], "sql": "..."})

print(f"{len(REPOSITORY)} queries loaded:")
for q in REPOSITORY:
    print(" -", q["id"])

9 queries loaded:
 - monthly_revenue
 - top_products_by_revenue
 - revenue_by_country
 - top_customers_by_spend
 - average_order_value
 - orders_by_status
 - category_sales
 - new_customers_by_year
 - product_catalog


## 4. The agent

Each question goes through three steps:

1. `match_query`: the LLM chooses the best repository query, or returns `null`.
2. `run_query`: the chosen SQL runs read-only against the database.
3. `answer_from_rows`: the LLM writes the answer from the returned rows only.

In [4]:
def get_client():
    """Create the OpenAI client on first use, so the notebook loads even without an API key."""
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("No API key set. Run the setup cell and paste your OpenAI API key.")
    return openai.OpenAI()


@dataclass
class QueryResult:
    query_id: str | None
    sql: str | None
    columns: list
    rows: list
    answer: str


def _ask(llm, system, user, max_tokens=500):
    response = llm.chat.completions.create(
        model=MODEL,
        max_completion_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
    )
    return (response.choices[0].message.content or "").strip()


def match_query(llm, question, repository):
    """Ask the LLM which repository query best answers the question. Returns its id or None."""
    catalog = "\n".join(
        f"- id: {q['id']}\n  description: {q['description']}\n"
        f"  example questions: {' | '.join(q['example_questions'])}"
        for q in repository
    )
    system = (
        "You route user questions to pre-written SQL queries. "
        "Pick the single query whose description and example questions best match the user's intent. "
        "If none of the queries is a reasonable match, choose null. "
        'Reply with only JSON, like {"query_id": "<id>"} or {"query_id": null}.'
    )
    user = f"Available queries:\n{catalog}\n\nUser question: {question}"

    raw = _ask(llm, system, user, max_tokens=100)
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        return None
    query_id = json.loads(match.group(0)).get("query_id")
    valid_ids = {q["id"] for q in repository}
    return query_id if query_id in valid_ids else None


def answer_from_rows(llm, question, columns, rows):
    """Ask the LLM for a plain-English answer grounded only in the query results."""
    if not rows:
        table = "(no rows returned)"
    else:
        shown = rows[:MAX_ROWS_FOR_ANSWER]
        table = "\n".join([" | ".join(columns)] + [" | ".join(str(v) for v in row) for row in shown])
    system = (
        "You answer business questions using query results. "
        "Use only the data provided; do not invent numbers. "
        "Answer in a few clear sentences or a short list. Round money to two decimals."
    )
    user = f"Question: {question}\n\nQuery results:\n{table}"
    return _ask(llm, system, user, max_tokens=600)


def answer_question(question, llm=None, repository=None, db_path=None):
    llm = llm or get_client()
    repository = repository or REPOSITORY
    db_path = db_path or DB_PATH

    query_id = match_query(llm, question, repository)
    if query_id is None:
        return QueryResult(None, None, [], [],
                           "I couldn't find a pre-written query that answers that. "
                           "Try asking about revenue, products, customers, countries, categories, or orders.")

    sql = next(q["sql"] for q in repository if q["id"] == query_id)
    columns, rows = run_query(sql, db_path=db_path)
    answer = answer_from_rows(llm, question, columns, rows)
    return QueryResult(query_id, sql, columns, rows, answer)


def ask(question):
    """Ask a question and print the matched query, the SQL, the rows, and the answer."""
    result = answer_question(question)
    if result.query_id:
        print(f"[matched query: {result.query_id}]")
        print(f"[SQL: {result.sql}]")
        print("[rows:]")
        print(" | ".join(result.columns))
        for row in result.rows[:10]:
            print(" | ".join(str(v) for v in row))
        if len(result.rows) > 10:
            print(f"... {len(result.rows) - 10} more rows")
    print(f"\nAnswer: {result.answer}")
    return result

## 5. Try it

Run a single question. The cell needs a valid API key.

In [5]:
result = ask("Which country buys the most?")

[matched query: revenue_by_country]
[SQL: SELECT c.country, ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue FROM orders o JOIN customers c ON c.id = o.customer_id JOIN order_items oi ON oi.order_id = o.id WHERE o.status = 'delivered' GROUP BY c.country ORDER BY revenue DESC;]
[rows:]
country | revenue
Australia | 93540.0
USA | 91462.0
Canada | 81910.0
India | 67916.0
UK | 65040.0
Germany | 38880.0

Answer: Australia buys the most, with a revenue of $93,540.00.


In [6]:
ask("How many orders were cancelled?")

[matched query: orders_by_status]
[SQL: SELECT status, COUNT(*) AS order_count FROM orders GROUP BY status ORDER BY order_count DESC;]
[rows:]
status | order_count
delivered | 689
shipped | 71
cancelled | 40

Answer: There were 40 orders that were cancelled.


QueryResult(query_id='orders_by_status', sql='SELECT status, COUNT(*) AS order_count FROM orders GROUP BY status ORDER BY order_count DESC;', columns=['status', 'order_count'], rows=[('delivered', 689), ('shipped', 71), ('cancelled', 40)], answer='There were 40 orders that were cancelled.')

## 6. Chat

Type a question and press Enter. Type `exit` to stop.

The chat is off by default, so *Run All* does not wait for input. Set `RUN_CHAT = True` in the cell below, then run it.

In [7]:
RUN_CHAT = False  # set to True to start the chat loop

if RUN_CHAT:
    print("SQL Insight Chatbot (type 'exit' to quit)\n")
    while True:
        try:
            question = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print()
            break
        if not question:
            continue
        if question.lower() in {"exit", "quit"}:
            break
        ask(question)
        print()

## 7. Tests

These tests do not call the API. A stub client stands in for OpenAI, so they run without an API key. They check:
- every repository query runs
- the matcher accepts valid IDs and rejects invented ones
- the end-to-end flow works
- the database refuses writes

In [8]:
class StubClient:
    """Stands in for the OpenAI client. Returns the replies you give it, in order."""

    def __init__(self, replies):
        self.replies = list(replies)
        self.chat = SimpleNamespace(completions=SimpleNamespace(create=self._create))

    def _create(self, **kwargs):
        text = self.replies.pop(0)
        return SimpleNamespace(choices=[SimpleNamespace(message=SimpleNamespace(content=text))])


class RepositoryTests(unittest.TestCase):
    def setUp(self):
        self.repo = load_repository()
        self.db = Path(tempfile.mkdtemp()) / "test.db"
        build_sample_db(self.db)

    def test_every_repository_query_runs(self):
        for q in self.repo:
            with self.subTest(query=q["id"]):
                columns, _ = run_query(q["sql"], db_path=self.db)
                self.assertTrue(columns)

    def test_match_returns_valid_id(self):
        llm = StubClient(['{"query_id": "monthly_revenue"}'])
        self.assertEqual(match_query(llm, "sales per month?", self.repo), "monthly_revenue")

    def test_match_rejects_unknown_id(self):
        llm = StubClient(['{"query_id": "made_up_query"}'])
        self.assertIsNone(match_query(llm, "anything", self.repo))

    def test_match_handles_null(self):
        llm = StubClient(['{"query_id": null}'])
        self.assertIsNone(match_query(llm, "what is the weather?", self.repo))

    def test_answer_question_end_to_end(self):
        llm = StubClient(['{"query_id": "orders_by_status"}', "There are 689 delivered orders."])
        result = answer_question("How many orders are delivered?", llm=llm,
                                 repository=self.repo, db_path=self.db)
        self.assertEqual(result.query_id, "orders_by_status")
        self.assertIn("689", result.answer)
        self.assertEqual(result.columns, ["status", "order_count"])

    def test_no_match_skips_database(self):
        llm = StubClient(['{"query_id": null}'])
        result = answer_question("Tell me a joke", llm=llm, repository=self.repo, db_path=self.db)
        self.assertIsNone(result.query_id)
        self.assertIsNone(result.sql)


class ReadOnlyTests(unittest.TestCase):
    def setUp(self):
        self.db = Path(tempfile.mkdtemp()) / "ro.db"
        build_sample_db(self.db)

    def test_rejects_non_select(self):
        with self.assertRaises(ValueError):
            run_query("DELETE FROM orders", db_path=self.db)

    def test_write_inside_with_is_blocked_by_read_only_mode(self):
        with self.assertRaises(sqlite3.OperationalError):
            run_query("WITH x AS (SELECT 1) DELETE FROM orders", db_path=self.db)

        _, rows = run_query("SELECT COUNT(*) FROM orders", db_path=self.db)
        self.assertEqual(rows[0][0], 800)


suite = unittest.TestLoader().loadTestsFromTestCase(RepositoryTests)
suite.addTests(unittest.TestLoader().loadTestsFromTestCase(ReadOnlyTests))
unittest.TextTestRunner(verbosity=2).run(suite)

test_answer_question_end_to_end (__main__.RepositoryTests.test_answer_question_end_to_end) ... ok
test_every_repository_query_runs (__main__.RepositoryTests.test_every_repository_query_runs) ... ok
test_match_handles_null (__main__.RepositoryTests.test_match_handles_null) ... ok
test_match_rejects_unknown_id (__main__.RepositoryTests.test_match_rejects_unknown_id) ... ok
test_match_returns_valid_id (__main__.RepositoryTests.test_match_returns_valid_id) ... ok
test_no_match_skips_database (__main__.RepositoryTests.test_no_match_skips_database) ... ok
test_rejects_non_select (__main__.ReadOnlyTests.test_rejects_non_select) ... ok
test_write_inside_with_is_blocked_by_read_only_mode (__main__.ReadOnlyTests.test_write_inside_with_is_blocked_by_read_only_mode) ... ok

----------------------------------------------------------------------
Ran 8 tests in 0.801s

OK


<unittest.runner.TextTestResult run=8 errors=0 failures=0>